In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.pyplot as plt
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import pyarrow as pa

In [26]:
df_add_cart = dd.read_parquet("add_to_cart.parquet")
df_page_visit = dd.read_parquet("page_visit.parquet")
df_product_buy = dd.read_parquet("product_buy.parquet")
df_remove_from_cart = dd.read_parquet("remove_from_cart.parquet")
df_search_query = dd.read_parquet("search_query.parquet")


In [27]:
"""
Just selecting Columns client_id and timestamp
"""

df_page_visit = df_page_visit[["client_id", "timestamp"]]
df_add_cart = df_add_cart[["client_id", "timestamp"]]
df_product_buy = df_product_buy[["client_id", "timestamp"]]
df_remove_from_cart = df_remove_from_cart[["client_id", "timestamp"]]
df_search_query = df_search_query[["client_id", "timestamp"]]

In [28]:
"""
Changing the name of the columns so the merge dont colapse
"""
df_page_visit = df_page_visit.rename(columns={"timestamp" : "page_visit_timestamp"})
df_add_cart = df_add_cart.rename(columns={"timestamp" : "add_to_cart_timestamp"})
df_product_buy = df_product_buy.rename(columns={"timestamp" : "product_buy_timestamp"})
df_remove_from_cart = df_remove_from_cart.rename(columns={"timestamp" : "remove_from_cart_timestamp"})
df_search_query = df_search_query.rename(columns={"timestamp" : "search_query_timestamp"})

In [29]:
df_page_visit = df_page_visit.repartition(npartitions=50)
df_add_cart = df_add_cart.repartition(npartitions=50)
df_product_buy = df_product_buy.repartition(npartitions=50)
df_remove_from_cart = df_remove_from_cart.repartition(npartitions=50)
df_search_query = df_search_query.repartition(npartitions=50)



In [30]:
"""
Merging all the client_id timestamps to a list so is like this
Client_id : List(ts)s
"""
df_page_visit = df_page_visit.groupby("client_id").agg({
    "page_visit_timestamp": list
}).reset_index()

df_add_cart = df_add_cart.groupby("client_id").agg({
    "add_to_cart_timestamp": list
}).reset_index()

df_product_buy = df_product_buy.groupby("client_id").agg({
    "product_buy_timestamp": list
}).reset_index()

df_remove_from_cart = df_remove_from_cart.groupby("client_id").agg({
    "remove_from_cart_timestamp": list
}).reset_index()

df_search_query = df_search_query.groupby("client_id").agg({
    "search_query_timestamp": list
}).reset_index()


In [31]:
df_page_visit = df_page_visit.set_index("client_id")
df_add_cart = df_add_cart.set_index("client_id")
df_product_buy = df_product_buy.set_index("client_id")
df_remove_from_cart = df_remove_from_cart.set_index("client_id")
df_search_query = df_search_query.set_index("client_id")

In [32]:
"""
Joined all the Datasets to the same dataset
"""
merged = df_page_visit \
    .join(df_add_cart, how="outer") \
    .join(df_product_buy, how="outer") \
    .join(df_remove_from_cart, how="outer") \
    .join(df_search_query, how="outer")


In [33]:
schema = pa.schema({
    "page_visit_timestamp": pa.list_(pa.string()),
    "add_to_cart_timestamp": pa.list_(pa.string()),
    "product_buy_timestamp": pa.list_(pa.string()),
    "remove_from_cart_timestamp": pa.list_(pa.string()),
    "search_query_timestamp": pa.list_(pa.string()),
    "client_id": pa.int64()
})


In [35]:
with ProgressBar():
    merged.to_parquet("merged_dataset.parquet", schema=schema, overwrite=True)

[#################                       ] | 44% Completed | 42m 7sss

IOStream.flush timed out


[########################################] | 100% Completed | 62m 24s


In [41]:
import dask.dataframe as dd
merged = dd.read_parquet("merged_dataset.parquet")

print(merged.columns)
    

Index(['page_visit_timestamp', 'add_to_cart_timestamp',
       'product_buy_timestamp', 'remove_from_cart_timestamp',
       'search_query_timestamp'],
      dtype='object')
